In [1]:
import csv
import pandas as pd
import os
import numpy as np
from datetime import datetime


base_dir = os.getcwd()
data_dir = os.path.join(base_dir, "data")
reshape_dir = os.path.join(base_dir,"reshape")
metadata_dir = os.path.join(base_dir,"metadata")
output_dir = os.path.join(base_dir,"output")
merge_dir = os.path.join(base_dir,"merge")

In [2]:
def get_country_code(filename):
    return filename.replace("pse-","").replace("-2024.xls","")[-3::].upper()


def make_abbreviation(description):
    """Generates an abbreviation using the first letter of each word in the description"""
    tokens = description.split()
    letters = [i[0].upper() for i in tokens if str.isalpha(i)]
    return "".join(letters)

def get_year_row(data):
    """ Returns the row with year information by selecting the first non-empty row in the last column"""
    data = data.reset_index(drop=True)
    yr = data[data.columns[len(data.columns)-1]]
    year_row = yr[yr.map(pd.notnull)].index[0]
    print("year row is "+str(year_row))
    return data.iloc[year_row]

def get_year_column(year_row):
    """ Returns the column index of the first numeric column in year_row"""
    years = year_row.reset_index(drop=True)
    years_nn = years[years.map(pd.notnull)]
    types = years_nn.map(type)
    year_column = types[types==np.float64].index[0]
    return year_column

files = os.listdir(data_dir) 

for file in files:

    #filename = 'ForDistribution___AUS.xls'

    filepath = os.path.join(data_dir,file)

    country_code = get_country_code(file)
    sheet_names = pd.ExcelFile(filepath).sheet_names
    
    out_file = country_code+'_MPS'+ ".csv"
    out_data= pd.DataFrame()

    mps_sheet_names = [i for i in sheet_names if 'MPS' in i and "GCT" not in i]

    for sheet_name in mps_sheet_names:
        print(sheet_name)
        cc = sheet_name.split()[0]
        data = pd.read_excel(filepath,sheet_name)

        if True:
            data = data.set_index(data.columns[0],drop=True)
            drop_null = data[pd.notnull(data[data.columns[0]])]
            descriptions = drop_null[drop_null.columns[0]].dropna()
            drop_null['abbrev'] = list(map(make_abbreviation,descriptions))

            new_index = [drop_null.abbrev[i] if pd.isnull(drop_null.index[i]) else drop_null.index[i] for i in range(0,len(drop_null.index))]
            data_rs = drop_null.set_index(pd.Index(new_index))

            year_row = get_year_row(data)
            year_column = get_year_column(year_row)
            print(year_column)

            num_columns = len(data.columns)
            data_t = data_rs[data_rs.columns[(year_column):num_columns]].T.reset_index(drop=True)
            years = year_row[(year_column):num_columns].reset_index(drop=True)
            data_t['year'] = years
            data_t = data_t.iloc[:, 1:len(data_t.columns)]
            data_form = data_rs[['Unnamed: 2']].T.reset_index(drop=True)
            data_form = data_form.iloc[:, 1:len(data_form.columns)]
            data_form.columns = data_form.columns+'_form'

            data_unit = data_rs[['Unnamed: 4']].T.reset_index(drop=True)
            data_unit = data_unit.iloc[:, 1:len(data_unit.columns)]
            data_unit.columns = data_unit.columns+'_unit'

            data_t['key'] = 1
            data_form['key'] = 1
            data_unit['key'] = 1
            data_t = pd.merge(pd.merge(data_t, data_form, on=['key']),data_unit, on=['key'])
            data_t = data_t.drop('key', axis=1)

            data_t['commodity_code'] = cc
            data_t['country_code'] = country_code
            data_t['Source_File'] = file

            out_data = pd.concat([out_data,data_t],ignore_index=True)
            out_path = os.path.join(reshape_dir,out_file)
            out_data.to_csv(out_path, index=False)

    out_file = country_code+'_SCT'+ ".csv"
    out_data= pd.DataFrame()
    sct_sheet_names = [i for i in sheet_names if 'SCT' in i and "GCT" not in i]

    for sheet_name in sct_sheet_names:
        print(sheet_name)
        cc = sheet_name.split()[0]
        data = pd.read_excel(filepath,sheet_name)

        if True:
            data = data.set_index(data.columns[0],drop=True)
            drop_null = data[pd.notnull(data[data.columns[0]])]

            descriptions = drop_null[drop_null.columns[0]].dropna()
            drop_null['abbrev'] = list(map(make_abbreviation,descriptions))


            new_index = [drop_null.abbrev[i] if pd.isnull(drop_null.index[i]) else drop_null.index[i] for i in range(0,len(drop_null.index))]
            data_rs = drop_null.set_index(pd.Index(new_index))

            year_row = get_year_row(data)
            year_column = get_year_column(year_row)
            print(year_column)

            num_columns = len(data.columns)
            data_t = data_rs[data_rs.columns[(year_column):num_columns]].T.reset_index(drop=True)

            years = year_row[(year_column):num_columns].reset_index(drop=True)
            data_t['year'] = years
            data_t = data_t.iloc[:, 1:len(data_t.columns)]
            data_form = data_rs[['Unnamed: 4']].T.reset_index(drop=True)
            data_form = data_form.iloc[:, 1:len(data_form.columns)]
            data_form.columns = data_form.columns+'_form'

            data_unit = data_rs[['Unnamed: 5']].T.reset_index(drop=True)
            data_unit = data_unit.iloc[:, 1:len(data_unit.columns)]
            data_unit.columns = data_unit.columns+'_unit'

            data_t['key'] = 1
            data_form['key'] = 1
            data_unit['key'] = 1
            data_t = pd.merge(pd.merge(data_t, data_form, on=['key']),data_unit, on=['key'])
            data_t = data_t.drop('key', axis=1)


            data_t['commodity_code'] = cc
            data_t['country_code'] = country_code
            data_t['Source_File'] = file

            out_data = pd.concat([out_data,data_t],ignore_index=True)
            out_path = os.path.join(reshape_dir,out_file)
            out_data.to_csv(out_path, index=False)

    out_file = country_code+'_commodity'+ ".csv"
    out_data= pd.DataFrame()
    index_sheet_names = [i for i in sheet_names if 'Index' in i]

    for sheet_name in index_sheet_names:
        print(sheet_name)
        out_data = pd.read_excel(filepath,sheet_name)
        out_data.reset_index(inplace=True)
        out_data.columns.values[1]='commodity_code'
        out_data.columns.values[2]='commodity_label'
        out_data=out_data[['commodity_code','commodity_label']]
        out_data = out_data.dropna()
        out_data['commodity_code'] = out_data['commodity_code'].astype(str)
        out_data = out_data[out_data['commodity_code'].str.contains('MPS')]
        out_data[['commodity_code','MPS']] = out_data['commodity_code'].str.split(' ',expand=True)
        out_data = out_data[['commodity_label', 'commodity_code']]
        out_data['commodity_label'] = out_data.commodity_label.replace({'Non MPS commodities': 'NonMPS from Workbook'})
        t_df = pd.DataFrame({'country_code': country_code, 'commodity_code': ['TOTAL'], 'commodity_label': ['Total']})
        out_data['country_code'] = country_code
        out_data = out_data.append(t_df)

        out_data = out_data[['country_code', 'commodity_code', 'commodity_label']].reset_index(drop=True)
        out_path = os.path.join(metadata_dir,out_file)
        out_data.to_csv(out_path, index=False)


    out_file = country_code+'_TOTAL'+ ".csv"
    out_data= pd.DataFrame()
    total_sheet_names = [i for i in sheet_names if 'TOTAL' in i]

    for sheet_name in total_sheet_names:
        print(sheet_name)
        cc = sheet_name.split()[0]
        data = pd.read_excel(filepath,sheet_name)

        if True:
            data = data.set_index(data.columns[0],drop=True)
            drop_null = data[pd.notnull(data[data.columns[0]])]

            descriptions = drop_null[drop_null.columns[0]].dropna()
            drop_null['abbrev'] = list(map(make_abbreviation,descriptions))
            new_index = [drop_null.abbrev[i] if pd.isnull(drop_null.index[i]) else drop_null.index[i] for i in range(0,len(drop_null.index))]
            data_rs = drop_null.set_index(pd.Index(new_index))

            year_row = get_year_row(data)
            year_column = get_year_column(year_row)
            print(year_column)

            num_columns = len(data.columns)
            data_t = data_rs[data_rs.columns[(year_column):num_columns]].T.reset_index(drop=True)
            years = year_row[(year_column):num_columns].reset_index(drop=True)
            data_t['year'] = years
            data_t = data_t.iloc[:, 0:len(data_t.columns)]

            data_form = data_rs[['Unnamed: 29']].T.reset_index(drop=True)
            data_form = data_form.iloc[:, 0:len(data_form.columns)]
            data_form.columns = data_form.columns+'_form'

            data_unit = data_rs[['Unnamed: 30']].T.reset_index(drop=True)
            data_unit = data_unit.iloc[:, 0:len(data_unit.columns)]
            data_unit.columns = data_unit.columns+'_unit'

            data_t['key'] = 1
            data_form['key'] = 1
            data_unit['key'] = 1
            data_t = pd.merge(pd.merge(data_t, data_form, on=['key']),data_unit, on=['key'])
            data_t = data_t.drop('key', axis=1)

            data_t['commodity_code'] = cc
            data_t['country_code'] = country_code
            data_t['Source_File'] = file

            out_data = pd.concat([out_data,data_t],ignore_index=True)

            out_data = out_data[['VP','VP_form','VP_unit', 'VP1P','VP1P_form','VP1P_unit', 'MPS','MPS_form','MPS_unit',
                                 'PSE','PSE_form','PSE_unit','PNPC', 'PNPC_form', 'PNPC_unit','CNPC', 'CNPC_form', 'CNPC_unit',
                                 'year','commodity_code', 'country_code', 'Source_File']].reset_index(drop=True)

            out_path = os.path.join(reshape_dir,out_file)
            out_data.to_csv(out_path, index=False)


def concatenate(files,suffix,col):
    out_file = pd.DataFrame()
    
    for file in files:
        #print(file)
        data = pd.read_csv(os.path.join(reshape_dir,file),encoding="latin1")
        data = data.rename(columns={'TIP':'PSCT'})
        
        if suffix in file:
            #print(file)
            data = data[col]
            out_file = pd.concat([out_file,data],ignore_index=True)
        else:
            pass
    return out_file


WT MPS


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\4021668485.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_null['abbrev'] = list(map(make_abbreviation,descriptions))


year row is 4
5
MA MPS
year row is 4
5
SB MPS


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\4021668485.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_null['abbrev'] = list(map(make_abbreviation,descriptions))
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\4021668485.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_null['abbrev'] = list(map(make_abbreviation,descriptions))


year row is 4
5
SF MPS


...


5
XE SCT


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\4021668485.py:102: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_null['abbrev'] = list(map(make_abbreviation,descriptions))


year row is 2
5
Index


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\4021668485.py:160: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  out_data = out_data.append(t_df)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\4021668485.py:181: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_null['abbrev'] = list(map(make_abbreviation,descriptions))


TOTAL
year row is 2
44


In [3]:
oecd_all_column = list(pd.read_csv(os.path.join(base_dir, './mapping/OECD_all_column.txt')))
sct_col = list(pd.read_csv(os.path.join(base_dir, './mapping/sct_column.txt')))
total_col = list(pd.read_csv(os.path.join(base_dir, './mapping/total_column.txt')))
mps_col = list(pd.read_csv(os.path.join(base_dir, './mapping/mps_column.txt')))

In [4]:
files = os.listdir(reshape_dir) 

out_mps = concatenate(files,"MPS",mps_col)
out_mps = out_mps[mps_col]
out_mps_path = os.path.join(merge_dir,"oecd_mps.xlsx")
out_mps.to_excel(out_mps_path)

out_sct = concatenate(files,"SCT",sct_col)
out_sct = out_sct[sct_col]
out_sct_path = os.path.join(merge_dir,"oecd_sct.xlsx")
out_sct.to_excel(out_sct_path)

mps_sct = out_mps.merge(out_sct,on=['country_code','commodity_code','year','Source_File'],how='outer')

xe = mps_sct[mps_sct['commodity_code']=='XE']
xe_df = xe[["MPS","VP", "commodity_code", "country_code","Source_File", "year"]].reset_index(drop=True)
xe_df.rename(columns={'MPS':'XE_MPS', 'VP':'XE_VP'}, inplace=True)
xe_df.commodity_code.replace('XE', 'TOTAL', inplace=True)

mps_sct_xe = mps_sct.merge(xe_df,on=['country_code','commodity_code','year','Source_File'],how='outer')

In [5]:
currency = pd.read_csv(os.path.join(base_dir, "./mapping/currency_map.csv")).set_index("ISO3")["CURRENCY"].to_dict()
country_map = pd.read_csv(os.path.join(base_dir, "./mapping/country_map.csv")).set_index("ISO3")["CountryLabel"].to_dict()

In [6]:
ex_data = pd.DataFrame()
er_dir = os.path.join(base_dir,"other_data")

ex_rate = pd.read_excel(os.path.join(er_dir,"XratesMon2024.xlsx"), skiprows=1)

ex_rate = ex_rate[ex_rate['Data we publish online']==1]

ex_rate = ex_rate.drop(['Data we publish online','Unnamed: 1', 'Source', 'Comments'], axis=1)
exch = pd.melt(ex_rate,id_vars=['Country','Currency code','Currency name'],var_name="year",value_name="exchange_rate")
# exch = exch.replace("EU","E27")

exc_subset = exch[['Country','year','exchange_rate']]
exc_subset.head()

,Country,year,exchange_rate
0,ARG,1986,9.430317e-05
1,AUS,1986,1.495996e+00
2,BRA,1986,4.964167e-09
3,CAN,1986,1.389448e+00
4,CHE,1986,1.798438e+00


In [7]:
mp = {}

for row in exc_subset.values:
    mp[(row[0],row[1])] = row[2]
    
subset = mps_sct_xe[['country_code','year']]
key = pd.Series([tuple(x) for x in subset.values])

mps_sct_xe = mps_sct_xe.reset_index()
mps_sct_xe['Exchange Rate - Official']  = key.map(mp) 
mps_sct_xe['ER - Official, Source'] = "OECD"
mps_sct_xe['ER - Official, Unit'] = mps_sct_xe.country_code.map(currency)

outtotal = pd.read_csv(os.path.join(base_dir, "./mapping/outtotal.csv")).set_index("totalcols")["rename"].to_dict()
mps_sct_xe.rename(columns={'Exchange Rate - Official':'EX_Rate', 'ER - Official, Unit':'EX_Unit', 
                           'ER - Official, Source': 'EX_Source', }, inplace=True)

out_total = concatenate(files,"TOTAL",total_col)
out_total.rename(columns=outtotal, inplace=True)
out_total_path = os.path.join(merge_dir,"oecd_total.xlsx")
out_total.to_excel(out_total_path)

In [8]:
OECD_all = mps_sct_xe.merge(out_total,on=['country_code','commodity_code','year','Source_File'],how='outer')

# Renaming commodity code for China Exported and Imported Fruits and Vegetables
OECD_all.loc[(OECD_all['country_code'] == 'CHN') & (OECD_all['commodity_code'] == 'IF'), 'commodity_code'] = 'IFCHN'
OECD_all.loc[(OECD_all['country_code'] == 'CHN') & (OECD_all['commodity_code'] == 'XF'), 'commodity_code'] = 'XFCHN'
OECD_all.loc[(OECD_all['country_code']=='ISR') & (OECD_all['commodity_code']=='EP'), 'commodity_code'] = 'MN'
OECD_all.loc[(OECD_all['country_code'].isin(['IND','CHN','ISR','ZAF'])) & (OECD_all['commodity_code']=='GN'),'commodity_code']='PN'


def concatenate_comm(files,suffix,col):
    out_file = pd.DataFrame()
    
    for file in files:
        #print(file)
        data = pd.read_csv(os.path.join(metadata_dir,file),encoding="latin1")
        
        
        if suffix in file:
            #print(file)
            data = data[col]
            out_file = pd.concat([out_file,data],ignore_index=True)
        else:
            pass
    return out_file


files = os.listdir(metadata_dir) 
commodity_col = ['country_code','commodity_code','commodity_label']
out_commodity = concatenate_comm(files,"commodity",commodity_col)

# Renaming commodity code for China Exported and Imported Fruits and Vegetables
out_commodity.loc[(out_commodity['country_code'] == 'CHN') & (out_commodity['commodity_code'] == 'IF'), 'commodity_code'] = 'IFCHN'
out_commodity.loc[(out_commodity['country_code'] == 'CHN') & (out_commodity['commodity_code'] == 'XF'), 'commodity_code'] = 'XFCHN'

out_commodity.loc[(out_commodity['country_code'] == 'ISR') & (out_commodity['commodity_code'] == 'EP'), 'commodity_code'] = 'MN'

out_commodity.loc[(out_commodity['country_code'].isin(['IND','CHN','ISR','ZAF'])) & (out_commodity['commodity_code']=='GN'), 'commodity_code']='PN'
out_commodity.loc[(out_commodity['country_code'].isin(['IND','CHN','ISR','ZAF'])) & (out_commodity['commodity_label']=='Groundnuts'), 'commodity_label']='Peanuts'

out_commodity.loc[(out_commodity['country_code'].isin(['ISR'])) & (out_commodity['commodity_label']=='Easy peelers'), 'commodity_label']='Mandarin'

OECD_all = OECD_all.merge(out_commodity, on=['country_code', 'commodity_code'], how='outer')

OECD_all['country_label'] = OECD_all.country_code.map(country_map)


OECD_all['Sourcefile_Date'] = '10/18/2022'
#OECD_all =rest.append([rus, cri, phl], ignore_index=True)

OECD_all['Sourcefile_Date'] = pd.to_datetime(OECD_all['Sourcefile_Date'], format='%m/%d/%Y').dt.date
OECD_all['EX_Sourcefile_Date'] = '10/26/2022'
OECD_all['EX_Sourcefile_Date'] = pd.to_datetime(OECD_all['EX_Sourcefile_Date'], format='%m/%d/%Y').dt.date

OECD_all['PSE'] = OECD_all['PSE']*10e5
OECD_all['PSE_unit'] = OECD_all.country_code.map(currency)


OECD_all.loc[(OECD_all['country_code'] == 'JPN'), 'PSE'] = OECD_all.PSE*1000
OECD_all.loc[(OECD_all['country_code'] == 'KOR'), 'PSE'] = OECD_all.PSE*1000


# Replace fake 0 for VC with null 
vc_na = OECD_all[OECD_all['VC']==0]
vc_na['VC'] = vc_na['VC'].replace(0, np.nan)
OECD_all.iloc[vc_na.index] = vc_na

oecdall_columns= list(pd.read_csv(os.path.join(base_dir, './mapping/oecdall_columns.txt')))

OECD_all=OECD_all[oecdall_columns]
out_oecd_path = os.path.join(merge_dir,"oecd_all.xlsx")
OECD_all.to_excel(out_oecd_path, sheet_name='DATA', index=False)

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\2172702783.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vc_na['VC'] = vc_na['VC'].replace(0, np.nan)


In [9]:
sugarcountries =OECD_all[OECD_all.commodity_code=='RS']
sugarcountries.country_code.unique()

array(['AUS', 'BRA', 'CHE', 'CHL', 'CHN', 'COL', 'CRI', 'EU', 'GBR',
       'IDN', 'IND', 'JPN', 'MEX', 'PHL', 'RUS', 'TUR', 'UKR', 'USA',
       'VNM', 'ZAF'], dtype=object)

In [10]:
sugar_check = OECD_all[OECD_all.commodity_code=='RS']
sugar_check = sugar_check[['country_code','commodity_code','year','MPD','PP','RP']]
sugar_check['MPDneqPPmRP'] = np.where(sugar_check['MPD']!=(sugar_check['PP']-sugar_check['RP']),1,0)
sugar_check['gap_PpRpMpd'] = np.where(abs(sugar_check['PP']-sugar_check['RP']-sugar_check['MPD'])>.005*sugar_check['PP'], 1,0)
sugar_case=  sugar_check[sugar_check.gap_PpRpMpd==1]
sugar_case.country_code.unique()

array(['BRA', 'CHE', 'COL', 'CRI', 'EU', 'GBR', 'JPN', 'RUS', 'TUR',
       'UKR', 'USA', 'ZAF'], dtype=object)

In [11]:
files = os.listdir(reshape_dir)

out_total = concatenate(files,"TOTAL",total_col)

out_mps = concatenate(files,"MPS",mps_col)
out_mps = out_mps[mps_col]

out_sct = concatenate(files,"SCT",sct_col)
out_sct = out_sct[sct_col]

m = out_mps.merge(out_sct,on=['country_code','commodity_code','year','Source_File'],how='outer')
print(m.shape)

m.loc[(m['country_code'].isin(['IND','CHN','ISR','ZAF'])) & (m['commodity_code']=='GN'), 'commodity_code']='PN'

m = pd.concat([m, pd.DataFrame(columns = ['Note_0','Note_1','Note_2','Note_3','Note_4','Note_5','Note_6','Note_7', 
                                          'Note_8','Note_9','Note_10','Note_11','Note_12','Note_13','Note_14','Note_15',
                                          'Note_16'])])

m['Note_0'] = 0

def recompute_rp(data,country,commodity):
    """ Recomputes the reference price where condition is True"""
    comm = data[data.commodity_code==commodity]
    comm = comm[comm.country_code==country]
    comm['RP'] = comm.PP-comm.MPD    
    data.loc[comm.index] = comm
    #data['RP'] = np.where(condition,data.PP-data.MPD,data.RP)
    return data

def trade_status(data,label):
    #print(len(data))
    qc_notnull = data[pd.isnull(data['QC'])]
    qp_isnull = data[pd.isnull(data['QP'])]
    data[label] = np.where(data.QC>data.QP,"iMports","eXports")
    nt = data[data.QC==data.QP]
    nt[label] = "Non Tradeable"
    data.loc[nt.index] = nt
    #print(len(data))
    return data

def exchange_rate_map(data):
    """ Creates dictionary from country, year to exchange rate value """
    mp = {}
    for row in data.values:
        mp[(row[0],row[1])] = row[2]
    return mp

##### Data Treatment #####

#2 Convert consumption quantity '000 tons to tonnes
m['QC'] = m['QC']*1000
#3 Convert MPS from million LCU to LCU
m['MPS'] = m['MPS']*1000000
#4 Convert PSCT from million LCU to LCU
m['PSCT'] = m['PSCT']*1000000

m.loc[(m['country_code'] == 'JPN'), 'MPS'] = m.MPS*1000
m.loc[(m['country_code'] == 'KOR'), 'MPS'] = m.MPS*1000
m.loc[(m['country_code'] == 'JPN'), 'PSCT'] = m.PSCT*1000
m.loc[(m['country_code'] == 'KOR'), 'PSCT'] = m.PSCT*1000

# Turkey production quantity unit issue ('000')
tur_qp = m[m.country_code=="TUR"]
tur_qp['QP'] = tur_qp['QP']/1000
m.loc[tur_qp.index] = tur_qp

  
#5 Countries where reference price for sugar was recalculated
rs_countries = ["USA","EU","BRA","CRI","RUS","TUR", "JPN", "CHE","UKR","COL","GBR"]
commodity = ['RS']
for country in rs_countries:
    m = recompute_rp(m,country,"RS")
    
    
for country in rs_countries:    
    rs = m[m.commodity_code=="RS"]
    rs = rs[rs.country_code==country]
    rs['Note_0'] = rs['Note_0'].replace(0, np.nan)
    rs['Note_3'] = "3"   
    m.loc[rs.index] = rs    
    if country=='USA':         
        rs['Note_15'] = "15"
        m.loc[rs.index] = rs 
    elif country=='BRA':              
        rs['Note_16'] = "16"
        m.loc[rs.index] = rs 
    elif country in ['EU','JPN']:            
        rs['Note_14'] = "14"
        m.loc[rs.index] = rs 


gbr = m[m.country_code=='GBR']
gbr['RP'] = gbr.PP-gbr.MPD    
m.loc[gbr.index] = gbr


m = m[ ~((m.country_code=='GBR') & (m.commodity_code=='RS') & (m.year==2021))]
m = m[ ~((m.country_code=='BRA') & (m.commodity_code=='CF') & (m.year==2023))]

print(m.shape)


(15474, 40)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\1076699825.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tur_qp['QP'] = tur_qp['QP']/1000


(15472, 57)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\1076699825.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gbr['RP'] = gbr.PP-gbr.MPD


In [12]:
#8 South Africa Sunflower
m = recompute_rp(m,"ZAF","SF")
m = recompute_rp(m,"ZAF","WT")

# South Africa Changes (Sunflower) for note 3
zaf_sfwf = m[(m.country_code=="ZAF") & (m.commodity_code.isin(["SF","WT"]))]
zaf_sfwf['Note_0'] = zaf_sfwf['Note_0'].replace(0, np.nan)
zaf_sfwf['Note_3'] = "3"
m.loc[zaf_sfwf.index] = zaf_sfwf

# South Africa Changes (Sunflower) for Note 4
zaf_sfwf = m[(m.country_code=="ZAF") & (m.commodity_code.isin(["SF","WT"])) & (m.MPD==0)]
zaf_sfwf['Note_3'] = zaf_sfwf['Note_3'].replace('3', np.nan)
zaf_sfwf['Note_4'] = "4"
m.loc[zaf_sfwf.index] = zaf_sfwf

# United Kingdom Changes (for commodities) for note 3
gbr_com = m[m.country_code=="GBR"]
gbr_com['Note_0'] = gbr_com['Note_0'].replace(0, np.nan)
gbr_com['Note_3'] = "3"
m.loc[gbr_com.index] = gbr_com

# United Kingdom Changes (for commodities where MPD=0) for Note 4
gbr_com = m[((m.country_code=="GBR") & (m.MPD==0))]
gbr_com['Note_3'] = gbr_com['Note_3'].replace('3', np.nan)
gbr_com['Note_4'] = "4"
m.loc[gbr_com.index] = gbr_com

# South Africa Changes (Sugarcane) ???
zaf_rs = m[(m.country_code=="ZAF") & (m.commodity_code=="RS")]
zaf_rs['Note_0'] = zaf_rs['Note_0'].replace(0, np.nan)
zaf_rs['Note_16'] = "16"
m.loc[zaf_rs.index] = zaf_rs

## Issue no 5 and 10 in notes. What note to be added here?
# Flowers
eu_fl = m[(m['country_code']=="EU") & (m['commodity_code'] == "FL")]
# China IF 
chn_if = m[(m['country_code'] == "CHN") & (m['commodity_code'] == "IF")]
chn_if['commodity_code'] = 'IFCHN'
# China XF
chn_xf = m[(m['country_code'] == "CHN") & (m['commodity_code'] == "XF")]
chn_xf['commodity_code'] = 'XFCHN'
ind_op = m[(m['country_code'] == "IND") & (m['commodity_code'] == "OP")]

arg_fv = m[(m['country_code'] == "ARG") & (m['commodity_code'] == "FV")]

list_df =[eu_fl,chn_if,chn_xf, ind_op, arg_fv]

for index, df in enumerate(list_df):
    df['QP'] = df.VP
    df['PP']=1
    df['RP'] = 1 - (df.MPS/(df.VP*10e5))
    df['Note_0']=df['Note_0'].replace(0, np.nan)
    df['Note_3']="3"

    m.loc[df.index]=df
    
# South Africa - MA. This holds this year too. 
zaf_ma = m[(m['country_code'] == "ZAF") & (m['commodity_code'] == "MA")]
zaf_ma['RP'] = zaf_ma.PP - ((zaf_ma.MPS)/(zaf_ma.QP*10e2))
zaf_ma['Note_0'] = zaf_ma['Note_0'].replace(0, np.nan)
zaf_ma["Note_3"] = "3"
m.loc[zaf_ma.index] = zaf_ma

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\1128344707.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  zaf_sfwf['Note_0'] = zaf_sfwf['Note_0'].replace(0, np.nan)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\1128344707.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  zaf_sfwf['Note_3'] = "3"
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\1128344707.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = 

In [13]:
# Japan
jpn = m[(m['country_code'] == "JPN") & (m['commodity_code']!='XE')]
jpn['VP'] = jpn.QP * jpn.PP /1000
m.loc[jpn.index] = jpn

# jpn_xe = m[(m['country_code']=='JPN') & (m['commodity_code']=='XE')]
# jpn_xe['VP'] = jpn_xe.VP*10e8
# m.loc[jpn_xe.index] = jpn_xe

# Japan
kor = m[(m['country_code'] == "KOR") & (m['commodity_code']!='XE')]
kor['VP'] = kor.QP * kor.PP/1000
m.loc[kor.index] = kor

# kor_xe = m[(m['country_code']=='KOR') & (m['commodity_code']=='XE')]
# kor_xe['VP'] = kor_xe.VP*10e8
# m.loc[kor_xe.index] = kor_xe

ind = m[((m['country_code']=='IND') & (m['commodity_code'].isin(['ON','MG','PO','TM'])))]
ind['VP'] = ind.PP*ind.QP/1000
m.loc[ind.index] = ind

zaf = m[((m['country_code']=='ZAF') & (m['commodity_code'].isin(['WT','MA','SF'])))]
zaf['VP'] = zaf.PP*zaf.QP/1000
m.loc[zaf.index] = zaf


m['rec_MPD'] = m.PP-m.RP

# CASE E
mps_gr_zero = m['MPS'] > 0
mpd_ls_zero = m['rec_MPD'] < 0
#efc_eq_zero = m['EFC'] == 0
efc_geq_zero = m['EFC'] >=0
case_e = m[mps_gr_zero & mpd_ls_zero & efc_geq_zero]
case_e['RP'] = case_e.PP-case_e.MPD
case_e['Note_0'] = case_e['Note_0'].replace(0, np.nan)
case_e['Note_3'] = "3"

case_e_e =  m[~(mps_gr_zero & mpd_ls_zero & efc_geq_zero)]
m = case_e_e.append(case_e)

m.shape

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\2727350092.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jpn['VP'] = jpn.QP * jpn.PP /1000
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\2727350092.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kor['VP'] = kor.QP * kor.PP/1000
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\2727350092.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

S

(15472, 58)

In [14]:
# Recomputed total
m_xe = m[m.commodity_code!="XE"]

vp_total = m_xe.groupby(["country_code","year"],as_index=False).VP.sum()
mps_total = m_xe.groupby(["country_code","year"],as_index=False).MPS.sum()

vp_share = out_total.groupby(["country_code","year"],as_index=False).VP1P.sum()
vp_share['VP1P'] = vp_share.VP1P/100

# Average MPS weighted by value of production
avg_mps = vp_total.merge(mps_total,on=['country_code','year'])
avg_mps['avg_MPS'] = avg_mps.MPS/avg_mps.VP


In [15]:
# Generated Non-MPS
gen_nonmps = avg_mps.merge(vp_share,on=["country_code","year"])
gen_nonmps['rec_VP'] = gen_nonmps.VP/gen_nonmps.VP1P
gen_nonmps['VP_NONMPS'] = (gen_nonmps.rec_VP-gen_nonmps.VP)
gen_nonmps['rec_NMPS'] = gen_nonmps.VP*((1-gen_nonmps.VP1P)/(gen_nonmps.VP1P))
gen_nonmps['rec_NMPS'] = gen_nonmps.rec_NMPS*10e5

gen_nonmps.loc[(gen_nonmps['country_code'] == 'JPN'), 'rec_NMPS'] = gen_nonmps.rec_NMPS*1000
gen_nonmps.loc[(gen_nonmps['country_code'] == 'KOR'), 'rec_NMPS'] = gen_nonmps.rec_NMPS*1000

gen_nonmps['MPS_NONMPS'] = gen_nonmps.VP_NONMPS*gen_nonmps['avg_MPS']
gen_nonmps['PSCT'] = gen_nonmps['MPS_NONMPS']
gen_nonmps['PP'] = 1
gen_nonmps['RP'] = 1-(gen_nonmps['MPS_NONMPS']/(gen_nonmps['rec_NMPS']))
gen_nonmps.head()

,country_code,year,VP,MPS,avg_MPS,VP1P,rec_VP,VP_NONMPS,rec_NMPS,MPS_NONMPS,PSCT,PP,RP
0,ARG,1986.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,1,NaN
1,ARG,1987.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,1,NaN
2,ARG,1988.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,1,NaN
3,ARG,1989.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,1,NaN
4,ARG,1990.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,1,NaN


In [16]:
# MPS comes from data in total sheet
rec_total = gen_nonmps
rec_total['QP'] = rec_total.rec_VP*10e5
rec_total.loc[(rec_total['country_code'] == 'JPN'), 'QP'] = rec_total.QP*1000
rec_total.loc[(rec_total['country_code'] == 'KOR'), 'QP'] = rec_total.QP*1000

rec_total['commodity_code'] = "TOTAL"
rec_total['Note_1'] = "1"
rec_total['Note_2'] = "2"
rec_total['Note_3'] = "3"
rec_total['Note_9'] = "9"

vp_TOTAL = rec_total[['rec_VP', 'country_code', 'commodity_code', 'year']]
vp_TOTAL.rename(columns={'rec_VP':'VP'}, inplace=True)

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\633472821.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vp_TOTAL.rename(columns={'rec_VP':'VP'}, inplace=True)


In [17]:
ot = out_total[['country_code','year','MPS','PSE','PSE_form','PSE_unit','VP1P_form','VP1P_unit']]
recomp_total = rec_total.merge(ot,on=['country_code','year'])
recomp_total['MPS'] = recomp_total.MPS_y*10e5
recomp_total['PSCT'] = recomp_total.PSE *10e5

recomp_total.loc[(recomp_total['country_code'] == 'JPN'), 'MPS'] = recomp_total.MPS*1000
recomp_total.loc[(recomp_total['country_code'] == 'KOR'), 'MPS'] = recomp_total.MPS*1000
recomp_total.loc[(recomp_total['country_code'] == 'JPN'), 'PSCT'] = recomp_total.PSCT*1000
recomp_total.loc[(recomp_total['country_code'] == 'KOR'), 'PSCT'] = recomp_total.PSCT*1000

recomp_total = recomp_total[['country_code','year','commodity_code','QP','RP','PP','MPS','PSCT', 
                             'PSE', 'PSE_form','PSE_unit','VP1P','VP1P_form','VP1P_unit', 'Note_1','Note_2','Note_3','Note_9']]

In [18]:
gen_nonmps['QP'] = gen_nonmps.rec_NMPS
gen_nonmps['Note_1'] = "1"
gen_nonmps['Note_2'] = "2"
gen_nonmps['Note_3'] = "3"
gen_nonmps['Note_9'] = "9"
gen_nonmps['commodity_code'] = "NONMPS"
gen_nonmps['MPS'] = gen_nonmps['MPS_NONMPS']
gen_nonmps['PSCT'] = gen_nonmps['MPS']
gen_nonmps = gen_nonmps[['country_code','year','commodity_code','QP','RP','PP','VP_NONMPS', 'MPS','PSCT','Note_1','Note_2',
                         'Note_3','Note_9']]
gen_nonmps.rename(columns={'VP_NONMPS': 'VP'}, inplace=True)

# gen_nonmps.to_csv('gen_nonmps2.csv')

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\423490620.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gen_nonmps.rename(columns={'VP_NONMPS': 'VP'}, inplace=True)


In [19]:
m_out = pd.concat([m,gen_nonmps])
m_out = pd.concat([m_out,recomp_total])
m_out = m_out.reset_index()

# Unit conversions
m_out_all = m_out[~(m_out.commodity_code.isin(['TOTAL','NONMPS']))]
m_out_all['QP'] = m_out_all.QP*1000
m_out_ex = m_out[m_out.commodity_code.isin(['TOTAL','NONMPS'])]

m_out = m_out_all.append(m_out_ex)
m_out.shape

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\551708301.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  m_out_all['QP'] = m_out_all.QP*1000
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\551708301.py:10: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  m_out = m_out_all.append(m_out_ex)


(17538, 65)

In [20]:
eu_fl = m_out[(m_out['country_code']=="EU") & (m_out['commodity_code'] == "FL")]
chn_if = m_out[(m_out['country_code'] == "CHN") & (m_out['commodity_code'] == "IFCHN")]
chn_xf = m_out[(m_out['country_code'] == "CHN") & (m_out['commodity_code'] == "XFCHN")]
ind_op = m_out[(m_out['country_code'] == "IND") & (m_out['commodity_code'] == "OP")]
arg_fv = m_out[(m_out['country_code'] == "ARG") & (m_out['commodity_code'] == "FV")]

list_df = [chn_if, chn_xf, eu_fl, ind_op, arg_fv]

for index, df in enumerate(list_df):
    df['QP'] = df.QP*1000
    m_out.loc[df.index] = df

m_out['EFC'] = m_out.EFC*1000000
m_out.loc[(m_out['country_code'] == 'JPN'), 'EFC'] = m_out.EFC*1000
m_out.loc[(m_out['country_code'] == 'KOR'), 'EFC'] = m_out.EFC*1000

m_out = trade_status(m_out,"Trade_Status")
m_out = m_out.merge(vp_TOTAL, on=['country_code', 'commodity_code', 'year'], how='left')
m_out.shape

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\427374309.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['QP'] = df.QP*1000
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\1076699825.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nt[label] = "Non Tradeable"


(17538, 67)

In [21]:
tot = m_out[m_out['commodity_code'] == "TOTAL"]
tot = tot.drop(['VP_x'], axis=1)
tot.rename(columns={'VP_y':'VP'},inplace=True)

totcolumns= list(pd.read_csv(os.path.join(base_dir, './mapping/totcolumns.txt')))
tot = tot[totcolumns]

In [22]:
m_out = m_out[m_out['commodity_code']!= "TOTAL"]
m_out = m_out.drop(['VP_y'], axis=1)
m_out.rename(columns={'VP_x':'VP'}, inplace=True)
m_out = m_out[totcolumns]

print(m_out.shape)
m_out = m_out.append(tot)
m_out.shape

(16505, 66)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\2657988482.py:7: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  m_out = m_out.append(tot)


(17538, 66)

In [23]:
########## Converting names to compare with previous input files

products = pd.read_csv(os.path.join(base_dir, "./mapping/commodity_map.csv")).set_index("commodity_code")["commodity_label"].to_dict()

# AgPolicyIndicators
ap_indicators = {
"country_code":"country_code",
"year":"year","commodity_code":"Product_Code",
"PP":"Producer price (farm gate)",
"MPS":"Market Price Support (LCU)",
"PCST":"Producer Single Commodity Transfers",
"QP":"Production_Quantity","QC":"Consumption_Quantity",
"RP":"Reference_Price (Farm Gate)",
}

In [24]:
# Korea units
kor = m_out[m_out.country_code=="KOR"]
jpn = m_out[m_out.country_code=="JPN"]

list_df = [kor, jpn]

col_list = ['CP','MPD','PP','RP','VP','VC','rec_MPD']

for index, df in enumerate(list_df):
    for c in col_list:
        if c in ['VP','VC']:
            df[c]=df[c]*10e8
        else:
            df[c]=df[c]*10e2
    m_out.loc[df.index] = df

col_list = ['PP_unit','CP_unit','RP_unit','MPD_unit']

for c in col_list:
    kor[c] = 'KRW/t'
    
for c in col_list:
    jpn[c] = 'JPY/t'

m_out.loc[kor.index] = kor
m_out.loc[jpn.index] = jpn
print(m_out.shape)

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\3192763330.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c]=df[c]*10e2
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\3192763330.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c]=df[c]*10e8


(17538, 66)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\3192763330.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kor[c] = 'KRW/t'
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\3192763330.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jpn[c] = 'JPY/t'


In [25]:
# Replace fake 0 for VC with null 
vc_na = m_out[m_out['VC']==0]
vc_na['VC'] = vc_na['VC'].replace(0, np.nan)
m_out.iloc[vc_na.index] = vc_na


# Replacing fake 0 of Australia Cotton with null after 2004
aus_ctn = m_out[(m_out['country_code']=="AUS") & (m_out['commodity_code']=="CT") & (m_out['year']>=2004)]
aus_ctn['QC'] = aus_ctn['QC'].replace(0, np.nan)
m_out.iloc[aus_ctn.index] = aus_ctn


# Replacing fake 0 for Israel Fruits and Vegetables with null before 1995 
isr_fv = m_out[(m_out['country_code']=="ISR") & (m_out['commodity_code']=='FV') & (m_out['year']<=1994)]

arg_yr = m_out[(m_out['country_code']=="ARG") & (m_out['year']==1996)]

phl_yr = m_out[(m_out['country_code']=="PHL") & (m_out['year']<2000)]

ind_yr = m_out[(m_out['country_code']=="IND") & (m_out['year']<2000)]

col_list = ['CP','EFC','MPD','MPS','CNPC','PP','PSCT','QC','QP','RP','VP','rec_MPD']

list_df = [isr_fv, phl_yr, ind_yr, arg_yr]

for index, df in enumerate(list_df):
    for c in col_list:
        df[c] = df[c].replace(0, np.nan)
    m_out.loc[df.index] = df


# Remove Mexico data before 1991, Argentina before 1997, India<2000, Philippines<2000
m_out = m_out[~((m_out.country_code=="MEX") & (m_out.year<1991))]
m_out = m_out[~((m_out.country_code=="ARG") & (m_out.year<1997))]
m_out = m_out[~((m_out.country_code=="IND") & (m_out.year<2000))]
m_out = m_out[~((m_out.country_code=="PHL") & (m_out.year<2000))]

# Recompute RP = PP and add notes as "4"
m_out['gap_PP_RP_MPD'] = np.where(abs(m_out['PP']-m_out['RP']-m_out['MPD'])>.005*m_out['PP'], 1,0)
m_out.shape

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\178227490.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  vc_na['VC'] = vc_na['VC'].replace(0, np.nan)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\178227490.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  aus_ctn['QC'] = aus_ctn['QC'].replace(0, np.nan)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\178227490.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

(16787, 67)

In [26]:
print(m_out[m_out.gap_PP_RP_MPD==1].shape)
print(m_out[(m_out.gap_PP_RP_MPD==1) & (m_out.MPD==0)].shape)
print(m_out[~((m_out['gap_PP_RP_MPD']==1) & (m_out['MPD']==0))].shape)

(3032, 67)
(2879, 67)
(13908, 67)


In [27]:
gap_mpd_zero = m_out[(m_out['gap_PP_RP_MPD']==1) & (m_out['MPD']==0)]
gap_mpd_zero_excl= gap_mpd_zero[~(((gap_mpd_zero['country_code']=='ZAF') & (gap_mpd_zero['year']==2006) & 
                  (gap_mpd_zero['commodity_code']=='MA'))|((gap_mpd_zero['country_code']=='EU') &
                                            (gap_mpd_zero['commodity_code']=='FL'))|((gap_mpd_zero['country_code']=='CHN') &
                                            (gap_mpd_zero['commodity_code']=='IFCHN'))|((gap_mpd_zero['country_code']=='IND') &
                                            (gap_mpd_zero['commodity_code']=='OP')))]
gap_mpd_zero_excl['RP'] = gap_mpd_zero_excl['PP']
gap_mpd_zero_excl['Note_0'] = gap_mpd_zero_excl['Note_0'].replace(0, np.nan)
gap_mpd_zero_excl['Note_4'] = "4"


gap_mpd_nzero = m_out[~((m_out['gap_PP_RP_MPD']==1) & (m_out['MPD']==0))]

gap_mpd_incl= gap_mpd_zero[(((gap_mpd_zero['country_code']=='ZAF') & (gap_mpd_zero['year']==2006) & 
                  (gap_mpd_zero['commodity_code']=='MA'))|((gap_mpd_zero['country_code']=='EU') &
                                            (gap_mpd_zero['commodity_code']=='FL'))|((gap_mpd_zero['country_code']=='CHN') &
                                            (gap_mpd_zero['commodity_code']=='IFCHN'))|((gap_mpd_zero['country_code']=='IND') &
                                            (gap_mpd_zero['commodity_code']=='OP')))]

m_out = gap_mpd_nzero.append([gap_mpd_zero_excl,gap_mpd_incl])
print(m_out.shape)

gap_mpd_ne = m_out[~(((m_out['gap_PP_RP_MPD']==1) & (m_out['MPD']>0)) | ((m_out['gap_PP_RP_MPD']==1) & (m_out['MPD']<0))) ]

gap_mpd_nzero = m_out[((m_out['gap_PP_RP_MPD']==1) & (m_out['MPD']>0)) | ((m_out['gap_PP_RP_MPD']==1) & (m_out['MPD']<0)) ]
gap_mpd_nzero['RP'] = gap_mpd_nzero['PP']-gap_mpd_nzero['MPD']
gap_mpd_nzero['Note_0'] = gap_mpd_nzero['Note_0'].replace(0, np.nan)
gap_mpd_nzero['Note_3'] = "3"

m_out = gap_mpd_ne.append(gap_mpd_nzero)
print(m_out.shape)

(16787, 67)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\3789212666.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gap_mpd_zero_excl['RP'] = gap_mpd_zero_excl['PP']
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\3789212666.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gap_mpd_zero_excl['Note_0'] = gap_mpd_zero_excl['Note_0'].replace(0, np.nan)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\3789212666.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.


(16787, 67)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\3789212666.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gap_mpd_nzero['Note_3'] = "3"
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\3789212666.py:30: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  m_out = gap_mpd_ne.append(gap_mpd_nzero)


In [28]:
ap_ind = m_out.rename(columns=ap_indicators)
ap_ind['Source'] = "OECD"
date = datetime.strftime(datetime.today(),"%m/%d/%Y")
ap_ind['Timestamp'] = date
ap_ind['country_label'] = ap_ind.country_code.map(country_map)
ap_ind.loc[(ap_ind['country_code'] == 'ISR') & (ap_ind['Product_Code'] == 'EP'), 'Product_Code'] = 'MN'
ap_ind['Product_Label (Farm Gate)'] = ap_ind.Product_Code.map(products)


# Convert production quantity to metric tonnes
ap_ind['Quantity_Physical_Unit'] = np.where(pd.notnull(ap_ind.Production_Quantity),"MT","")
ap_ind['Quantity_Physical_Unit'] = np.where(ap_ind["Product_Label (Farm Gate)"].isin(['Generated Non MPS','Total',
                                                                                     'NonMPS from Workbook']),"AG","MT")

ap_ind['Consumption_Physical_Unit'] = np.where(pd.notnull(ap_ind.Consumption_Quantity),"MT","AG")
ap_ind['Consumption_Physical_Unit'] = np.where(ap_ind["Product_Label (Farm Gate)"].isin(['Generated Non MPS','Total',
                                                                                     'NonMPS from Workbook']),"AG","MT")

ap_ind['Price_Physical_Unit'] = np.where(ap_ind['Producer price (farm gate)']==1,"AG","MT")

ap_ind['Currency Unit'] = ap_ind.country_code.map(currency)
print(ap_ind.shape)

(16787, 75)


In [29]:
################# 
# AgDistortions_metadata
###################
ag_distortions = {
"country_code":"country_code",
"commodity_code":"commodity_code",
"year":"year",
"QP":"Level of Production",
"QC":"Level of Consumption",
"RP":"Reference Price at the Farm Gate level",
"PP":"Farm Gate, Price",
"Trade_Status":"Commodity Trade Status",
}

ag_meta = m_out.rename(columns=ag_distortions)


ag_meta['country_label'] = ag_meta.country_code.map(country_map)
ag_meta.loc[(ap_ind['country_code'] == 'ISR') & (ag_meta['commodity_code'] == 'EP'), 'commodity_code'] = 'MN'
ag_meta['commodity_label'] = ag_meta.commodity_code.map(products)


ag_meta['Level of Production, Physical Unit'] = np.where(pd.notnull(ag_meta['Level of Production']),"MT","")
ag_meta['Level of Production, Physical Unit'] = np.where(ag_meta['commodity_code'].isin(['XE','NONMPS','TOTAL']),"AG","MT")
ag_meta['Level of Production, Source'] = np.where(pd.notnull(ag_meta['Level of Production']),"data","") 
ag_meta['Level of Production, Type'] = "Primary/Original data"


ag_meta['Level of Consumption, Physical Unit'] = np.where(pd.notnull(ag_meta['Level of Consumption']),"MT","AG")
ag_meta['Level of Consumption, Physical Unit'] = np.where(ag_meta['commodity_code'].isin(['XE','NONMPS','TOTAL']),"AG","MT")
ag_meta['Level of Consumption, Source'] = np.where(pd.notnull(ag_meta['Level of Consumption']),"data","") 
ag_meta['Level of Consumption, Type'] = "Primary/Original data"

# Exchange rates
exc_map = exchange_rate_map(exc_subset)

subset = ag_meta[['country_code','year']]
key = pd.Series([tuple(x) for x in subset.values])


ag_meta = ag_meta.reset_index()
ag_meta['Exchange Rate - Official']  = key.map(exc_map) 
ag_meta['ER - Official, Source'] = "OECD"
ag_meta['ER - Official, Unit'] = ag_meta.country_code.map(currency)

ag_meta['Farm Gate Price, Physical Unit'] = "MT" 



ap_ind = ap_ind.reset_index()
ap_ind['Exchange Rate - Official']  = key.map(exc_map) 
ap_ind['ER - Official, Source'] = "OECD"
ap_ind['ER - Official, Unit'] = ap_ind.country_code.map(currency)
print(ap_ind.shape)

# EU FL and Israel FV are AG.
eu_fl = ag_meta[(ag_meta['country_code'] == "EU") & (ag_meta['commodity_code'] ==  "FL")]
isr_fv = ag_meta[(ag_meta['country_code'] == "ISR") & (ag_meta['commodity_code'] == "FV")]
eu_fl['Farm Gate Price, Physical Unit'] = "AG"
isr_fv['Farm Gate Price, Physical Unit'] = "AG"
ag_meta.iloc[eu_fl.index] = eu_fl
ag_meta.iloc[isr_fv.index] = isr_fv

ag_meta['Farm Gate Price, Monetary Unit'] = ag_meta.country_code.map(currency)
ag_meta['Farm Gate Price, Type'] = "Computed data"
ag_meta['Farm Gate Price, Source'] = "Formula" # only Chile is data
chl = ag_meta[ag_meta['country_code'] == "CHL"]
chl['Farm Gate Price, Source'] = "Data"
ag_meta.iloc[chl.index] = chl

print(ag_meta.shape)

(16787, 79)
(16787, 83)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\2844530350.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eu_fl['Farm Gate Price, Physical Unit'] = "AG"
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\2844530350.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chl['Farm Gate Price, Source'] = "Data"


In [30]:
# Cookbook formula codes
cb_map = pd.read_excel(os.path.join(base_dir,"cookbook_mapping.xlsx"))
cb_map['Country'] = cb_map['Country'].str.upper()
cb_map = exchange_rate_map(cb_map)
sb_set = ag_meta[['country_label','commodity_code']]
sb_set_key = pd.Series([tuple(x) for x in sb_set.values])


In [31]:
ag_meta['Reference Price at FG, Physical Unit'] = "MT"
ag_meta['Reference Price at FG, Monetary Unit'] = ag_meta.country_code.map(currency)
ag_meta['Reference Price at FG, Methodology'] =  sb_set_key.map(cb_map) # Cookbook acronyms
print(ag_meta.shape)

(16787, 86)


In [32]:
missing_code = ag_meta[pd.isnull(ag_meta['Reference Price at FG, Methodology'])]
missing_code['Reference Price at FG, Methodology'] = "Missing Code"
ag_meta.iloc[missing_code.index] = missing_code

ag_meta['Reference Price at FG, Source'] = "Cookbook" # formula for refined sugar Chile E28 USA


# U.S. Changes
us = ag_meta[ag_meta.country_code=="USA"]
us['Reference Price at FG, Source'] = "Formula"
ag_meta.iloc[us.index] = us


# Chile Changes
chl = ag_meta[ag_meta.country_code=='CHL']
chl['Reference Price at FG, Source'] = "Formula"
ag_meta.iloc[chl.index] = chl


meta_map = pd.read_excel(os.path.join(base_dir,"mapping_headers.xlsx"),"agparameters").T.to_dict()[0]
ap_map = pd.read_excel(os.path.join(base_dir,"mapping_headers.xlsx"),"agPolicy").T.to_dict()[0]


ad_meta = ag_meta.rename(columns=meta_map)
ad_meta = ad_meta.drop(["EFC","MPS","MPD","PSCT","level_0"],axis=1)
ad_meta = ad_meta.replace("EU","EUR")


#ad_meta = ad_meta[ad_meta.commodity_code!="TOTAL"]
ad_meta = ad_meta.drop_duplicates()

metaunit = ad_meta[ad_meta['commodity_code'].isin(["FV","FL","IFCHN","XFCHN","OP"])]
list_df = [metaunit]
col_list = ['PROP_PHY_UNIT','CONSQ_PHY_UNIT','REFP_PHY_UNIT','PRODQ_PHY_UNIT']
for index, df in enumerate(list_df):
    for c in col_list:
        df[c] = 'AG'

    ad_meta.iloc[df.index]=df
    
col_fl = ad_meta[(ad_meta['country_code'] == "COL") & (ad_meta['commodity_code'] == "FL")]
list_df = [col_fl]
for index, df in enumerate(list_df):
    for c in col_list:
        df[c] = 'MT'

    ad_meta.iloc[df.index]=df

print(ad_meta.shape)

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\592787906.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_code['Reference Price at FG, Methodology'] = "Missing Code"
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\592787906.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  us['Reference Price at FG, Source'] = "Formula"
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\592787906.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

(16787, 82)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\592787906.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = 'AG'
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\592787906.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = 'MT'


In [33]:
admeta = list(pd.read_csv(os.path.join(base_dir, './mapping/admeta.txt')))
ad_meta = ad_meta[admeta]


# ad_meta = ad_meta[ad_meta.year>=1996]
print(ad_meta.shape)
ad_meta.to_csv(os.path.join(output_dir,"ad_meta.csv"),index=False)

agp_ind = ap_ind.rename(columns=ap_map)
agp_ind = agp_ind.replace("EU","EUR")
agp_ind = agp_ind.drop_duplicates()


agp_ind.rename(columns={'COMMODITY_CODE': 'commodity_code', 'COMMODITY_LABEL': 'commodity_label'}, inplace=True)


agp_ind['VC'] = agp_ind['VC']*1000000
agp_ind['VP'] = agp_ind['VP']*1000000
agp_ind['PSE'] = agp_ind['PSE']*10e5
agp_ind['PSE_unit'] = agp_ind.country_code.map(currency)

jpn = agp_ind[agp_ind['country_code'] == "JPN"]
kor = agp_ind[agp_ind['country_code'] == "KOR"]

col_list = ['VP','VC']
list_df = [jpn, kor]

for index, df in enumerate(list_df):
    for c in col_list:
        df[c] = df[c]/1000000
    df['PSE'] = df.PSE*1000

    agp_ind.iloc[df.index] = df

agp_ind.rename(columns={'Consumption_Physical_Unit': 'CONSQ_PHY_UNIT'}, inplace=True)

agp_ind.shape

(16787, 37)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\4105554043.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = df[c]/1000000
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\4105554043.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['PSE'] = df.PSE*1000


(16787, 79)

In [34]:
agpind = list(pd.read_csv(os.path.join(base_dir, './mapping/agpind.txt')))
ap_indicators = agp_ind[agpind]

print(ap_indicators.shape)
# ap_indicators = ap_indicators[ap_indicators.year>=1996]

agdistortions_data = ap_indicators.merge(ad_meta, on=['country_code', 'year', 'commodity_code'])

agdistortions_data.rename(columns={'country_code':'COUNTRY_CODE', 'country_label':'COUNTRY_LABEL', 
                                    'commodity_code': 'COMMODITY_CODE', 'commodity_label': 'COMMODITY_LABEL', 'year':'YEAR', 
                                    'MPD': 'MPD_SOURCE', 'CNPC': 'NPC_SOURCE'}, inplace=True)
print(agdistortions_data.shape)
agdistortions_data.NPC_SOURCE = np.where(agdistortions_data.NPC_SOURCE=="..","", agdistortions_data.NPC_SOURCE)
agdistortions_data.NPC_SOURCE = pd.to_numeric(agdistortions_data.NPC_SOURCE, errors='coerce')

# GBR data starts from 2017 and VNM from 2000
agdistortions_data = agdistortions_data[~((agdistortions_data.COUNTRY_CODE=='GBR') & (agdistortions_data.YEAR<2017))]
agdistortions_data = agdistortions_data[~((agdistortions_data.COUNTRY_CODE=='VNM') & (agdistortions_data.YEAR<2000))]

agdistortions_data.COUNTRY_LABEL = agdistortions_data.COUNTRY_LABEL.str.lower().str.title()

agdistortions_data.COUNTRY_LABEL = np.where(agdistortions_data.COUNTRY_LABEL=='Turkey', "Türkiye", agdistortions_data.COUNTRY_LABEL)


agdistortions_data.to_csv(os.path.join(output_dir,"OECD_input_file.csv"),index=False, encoding="utf-8-sig")


(16787, 20)
(16787, 54)


In [35]:
agp_xe = agp_ind[agp_ind['commodity_code']=='XE']
agp_xe_tot = agp_xe[["MPS","VP", "commodity_code", "country_code", "year"]].reset_index(drop=True)
agp_xe_tot.rename(columns={'MPS':'XE_MPS', 'VP':'XE_VP'}, inplace=True)
agp_xe_tot.commodity_code.replace('XE', 'TOTAL', inplace=True)


agp_xe_nonmps = agp_xe[["MPS","VP", "commodity_code", "country_code","year"]].reset_index(drop=True)
agp_xe_nonmps.rename(columns={'MPS':'XE_MPS', 'VP':'XE_VP'}, inplace=True)
agp_xe_nonmps.commodity_code.replace('XE', 'NONMPS', inplace=True)


agp_xe_df = agp_xe_tot.append(agp_xe_nonmps, ignore_index=True)



agp_total = agp_ind[agp_ind['commodity_code']=='TOTAL']
agp_total = agp_total[["MPS","VP", "commodity_code", "country_code","SOURCE_FILE", "year"]].reset_index(drop=True)
agp_total.rename(columns={'MPS':'TOT_MPS', 'VP':'TOT_VP'}, inplace=True)

agp_total_nonmps = agp_ind[agp_ind['commodity_code']=='TOTAL']
agp_total_nonmps = agp_total_nonmps[["MPS","VP", "commodity_code", "country_code","SOURCE_FILE", "year"]].reset_index(drop=True)
agp_total_nonmps.rename(columns={'MPS':'TOT_MPS', 'VP':'TOT_VP'}, inplace=True)
agp_total_nonmps.commodity_code.replace('TOTAL', 'NONMPS', inplace=True)

agp_total_df = agp_total.append(agp_total_nonmps, ignore_index=True)

agp_ind_df = agp_ind.merge(agp_xe_df,on=['country_code', 'year', 'commodity_code'], how='outer')
agp_ind_df = agp_ind_df.merge(agp_total_df,on=['country_code', 'year', 'commodity_code','SOURCE_FILE'],how='outer')



agp_ind_df['SOURCEFILE_DATE'] = '10/01/2022'

# Format date variable of source file to date type
agp_ind_df['SOURCEFILE_DATE'] = pd.to_datetime(agp_ind_df['SOURCEFILE_DATE'], format='%m/%d/%Y').dt.date

# Inserting date of exchange rate source file received and format to date type 
agp_ind_df['EX_SOURCEFILE_DATE'] = '10/01/2022'
agp_ind_df['EX_SOURCEFILE_DATE'] = pd.to_datetime(agp_ind_df['EX_SOURCEFILE_DATE'], format='%m/%d/%Y').dt.date


# changing unit values of quantity production and quantity consumed to MT
agp_ind_df['QP_unit'] = 'MT'
agp_ind_df['QC_unit'] = 'MT'

agpinddfunit = agp_ind_df[agp_ind_df['commodity_code'].isin(["FV","FL","IFCHN","XFCHN","OP","XE","NONMPS","TOTAL"])]
agpinddfunit['QP_unit'] = 'AG'
agpinddfunit['QC_unit'] = 'AG'
agp_ind_df.iloc[agpinddfunit.index] = agpinddfunit

col_fl = agp_ind_df[(agp_ind_df['country_code'] == "COL") & (agp_ind_df['commodity_code'] == "FL")]
col_fl['QP_unit'] = 'MT'
col_fl['QC_unit'] = 'MT'
agp_ind_df.iloc[col_fl.index] = col_fl


# Inserting unit values mapped with currency unit of respective country. 
agp_ind_df['MPS_unit'] = agp_ind_df.country_code.map(currency)
agp_ind_df['PSCT_unit'] = agp_ind_df.country_code.map(currency)
agp_ind_df['EFC_unit'] = agp_ind_df.country_code.map(currency)
agp_ind_df['VC_unit'] = agp_ind_df.country_code.map(currency)
agp_ind_df['VP_unit'] = agp_ind_df.country_code.map(currency)

agp_ind_df['EX_SOURCEFILE']= 'Exchange_rates.xlsx'


# Keeping the variables in order
additionalcolumns = list(pd.read_csv(os.path.join(base_dir, './mapping/additionaldatacolumns.txt')))
agp_ind_df = agp_ind_df[additionalcolumns]


### HERE doing all the checks that were done in VBA code 

sector_df = agp_ind_df[~agp_ind_df['commodity_code'].isin(['TOTAL', 'XE', 'NONMPS'])]
sum_mps = pd.DataFrame(sector_df.groupby(['country_code', 'year'])[['MPS']].sum()).reset_index()
sum_vp = pd.DataFrame(sector_df.groupby(['country_code', 'year'])[['VP']].sum()).reset_index()
sum_val = sum_mps.merge(sum_vp, on=['country_code', 'year'])
sum_val.rename(columns={'MPS': 'SUM(SEC_MPS)', 'VP': 'SUM(SEC_VP)'}, inplace=True)
sum_val['SUM(SEC_MPS)/SUM(SEC_VP)'] = sum_val['SUM(SEC_MPS)']/sum_val['SUM(SEC_VP)'].replace({0 : np.nan})


# Check if value of production equals production quantity times production price. In the second line, the delta is checked and allowed less than .001
agp_ind_df['IF(QP*PP=VP)'] = np.where(agp_ind_df['PRODQ']*agp_ind_df['PROP']==agp_ind_df['VP'], 1, 0)
agp_ind_df['IF(QP*PP~VP)'] = np.where(abs(agp_ind_df['PRODQ']*agp_ind_df['PROP']-agp_ind_df['VP'])<.001, 1, 0)

# Check if value of consumption equals consumption quantity times consumption price. In the second line, the delta is checked and allowed less than .001
agp_ind_df['IF(QC*CP=VC)'] = np.where(agp_ind_df['CONSQ']*agp_ind_df['CP']==agp_ind_df['VC'], 1, 0)
agp_ind_df['IF(QC*CP~VC)'] = np.where(abs(agp_ind_df['CONSQ']*agp_ind_df['CP']-agp_ind_df['VC'])<.001, 1, 0)

# Check if MPS generated and MPS from workbook of NONMPS commodities are equal. In the fourth line, the delta is checked and allowed less than .001
nonmps = agp_ind_df[agp_ind_df['commodity_code']=='NONMPS']
nonmps = nonmps[["MPS","XE_MPS", "XE_VP", "commodity_code", "country_code", "year"]].reset_index(drop=True)
nonmps['IF(MPS=XE_MPS)'] = np.where(nonmps['MPS']==nonmps['XE_MPS'], 1, 0)
nonmps['IF(MPS~XE_MPS)'] = np.where(abs(nonmps['MPS']-nonmps['XE_MPS'])<.001, 1, 0)
nonmps['(NONMPS_MPS/NONMPS_VP)'] = nonmps['MPS']/nonmps['XE_VP'].replace({ 0 : np.nan })
nonmps = nonmps[['(NONMPS_MPS/NONMPS_VP)', "IF(MPS=XE_MPS)","IF(MPS~XE_MPS)", "commodity_code", "country_code", "year"]].reset_index(drop=True)


sumMpsVp_nonmps = agp_ind_df[agp_ind_df['commodity_code']=='NONMPS']
sumMpsVp_nonmps = sumMpsVp_nonmps[["commodity_code", "country_code", "year"]].reset_index(drop=True)
sumMpsVp_nonmps = sumMpsVp_nonmps.merge(sum_val, on=['country_code', 'year'])

sumMpsVp_total = agp_ind_df[agp_ind_df['commodity_code']=='TOTAL']
sumMpsVp_total = sumMpsVp_total[["commodity_code", "country_code", "year"]].reset_index(drop=True)
sumMpsVp_total = sumMpsVp_total.merge(sum_val, on=['country_code', 'year'])

nonmps_total = sumMpsVp_nonmps.append(sumMpsVp_total)



# Check if VP generated and VP from workbook of NONMPS commodities are equal. In the fourth line, the delta is checked and allowed less than .001
vp = agp_ind_df[agp_ind_df['commodity_code']=='NONMPS']
vp = vp[["VP","XE_VP", "commodity_code", "country_code", "year"]].reset_index(drop=True)
vp['IF(VP=XE_VP)'] = np.where(vp['VP']==vp['XE_VP'], 1, 0)
vp['IF(VP~XE_VP)'] = np.where(abs(vp['VP']-vp['XE_VP'])<.001, 1, 0)
vp = vp[["IF(VP=XE_VP)","IF(VP~XE_VP)", "commodity_code", "country_code", "year"]].reset_index(drop=True)

# Merging files with the master sheet. 
agp_ind_df = agp_ind_df.merge(nonmps_total,on=['country_code', 'year', 'commodity_code'], how='outer')
agp_ind_df = agp_ind_df.merge(nonmps,on=['country_code', 'year', 'commodity_code'], how='outer')
agp_ind_df = agp_ind_df.merge(vp,on=['country_code', 'year', 'commodity_code'], how='outer')


# IF[(NONMPS MPS)/(NONMPS VP)]-[SUM(SECTOR MPS)/SUM(SECTOR VP)=0] 
mps_vp = agp_ind_df[agp_ind_df['commodity_code'].isin(['TOTAL', 'NONMPS'])]
mps_vp = mps_vp[['(NONMPS_MPS/NONMPS_VP)', 'SUM(SEC_MPS)', 'SUM(SEC_MPS)/SUM(SEC_VP)', 'country_code', 'commodity_code', 'year']].reset_index(drop=True)
mps_vp['(NONMPS_MPS/NONMPS_VP)~SUM(SEC_MPS)/SUM(SEC_VP)'] = np.where((mps_vp['(NONMPS_MPS/NONMPS_VP)']-
                                                                            mps_vp['SUM(SEC_MPS)/SUM(SEC_VP)'])<.0000004, 1, 0)

mps_vp_nonzero = mps_vp[mps_vp['SUM(SEC_MPS)']!=0]

mps_vp_nonzero = mps_vp_nonzero[["(NONMPS_MPS/NONMPS_VP)","SUM(SEC_MPS)/SUM(SEC_VP)", "country_code", "commodity_code", "year"]].reset_index(drop=True)

mps_vp_nonzero["(NONMPS_MPS/NONMPS_VP)-SUM(SEC_MPS)/SUM(SEC_VP)"] = mps_vp_nonzero['(NONMPS_MPS/NONMPS_VP)']-mps_vp_nonzero['SUM(SEC_MPS)/SUM(SEC_VP)']

mps_vp_nonzero =  mps_vp_nonzero[["(NONMPS_MPS/NONMPS_VP)-SUM(SEC_MPS)/SUM(SEC_VP)", "country_code", "commodity_code", "year"]].reset_index(drop=True)

mps_vp = mps_vp.merge(mps_vp_nonzero, on=['country_code', 'commodity_code', 'year'], how='outer')


mps_vp = mps_vp[['(NONMPS_MPS/NONMPS_VP)~SUM(SEC_MPS)/SUM(SEC_VP)', '(NONMPS_MPS/NONMPS_VP)-SUM(SEC_MPS)/SUM(SEC_VP)','country_code','commodity_code', 'year']].reset_index(drop=True)


agp_ind_df = agp_ind_df.merge(mps_vp, on=['country_code', 'commodity_code', 'year'], how='outer')

# Check if MPS and VP are equal to the computed total MPS and VP 
totalled = agp_ind_df[~agp_ind_df['commodity_code'].isin(['TOTAL','XE'])]
totalled = totalled[['MPS', 'VP', 'country_code', 'commodity_code', 'year']].reset_index(drop=True)
totalled_mps = pd.DataFrame(totalled.groupby(['country_code', 'year'])[['MPS']].sum()).reset_index()
totalled_vp = pd.DataFrame(totalled.groupby(['country_code', 'year'])[['VP']].sum()).reset_index()
totalled = totalled_mps.merge(totalled_vp, on=['country_code', 'year'])
totalled.rename(columns={'MPS': 'TOTALLED_MPS', 'VP': 'TOTALLED_VP'}, inplace=True)

total =agp_ind_df[agp_ind_df['commodity_code']=='TOTAL']
total = total[['TOT_MPS', 'VP','VP1P','country_code', 'commodity_code', 'year']].reset_index(drop=True)


totalled = totalled.merge(total, on=['country_code', 'year'], how='outer')
totalled['TOTAL_MPS-TOTALLED_MPS'] = np.where(totalled['TOT_MPS']-totalled['TOTALLED_MPS']<.001, 1, 0)
totalled['TOTAL_VP-TOTALLED_VP'] = np.where(totalled['VP']-totalled['TOTALLED_VP']<.001,1,0)
totalled['Total_VP*(1-Share)'] = totalled['TOTALLED_VP']*(1-totalled['VP1P'])
totalled = totalled[['TOTAL_MPS-TOTALLED_MPS','TOTAL_VP-TOTALLED_VP','Total_VP*(1-Share)','country_code','commodity_code','year']].reset_index(drop=True)

agp_ind_df = agp_ind_df.merge(totalled, on=['country_code', 'commodity_code', 'year'], how='outer')

agp_ind_df['gap_pp_rp_mpd'] = agp_ind_df['PROP']-agp_ind_df['REFP']-agp_ind_df['MPD']
agp_ind_df['MPD_neq_PP_RP'] = np.where(abs(agp_ind_df['PROP']-agp_ind_df['REFP']-agp_ind_df['MPD'])>.005*agp_ind_df['PROP'], 1,0)

agp_ind_df = agp_ind_df[agp_ind_df.year>=1996]

agp_ind_df.country_label = agp_ind_df.country_label.str.lower().str.title()

agp_ind_df.country_label = np.where(agp_ind_df.country_label=='Turkey', "Türkiye", agp_ind_df.country_label)

path = r"C:/Users/AMAMUN/OneDrive - CGIAR/Documents/AgIncentives/OECD_preprocessing/Update_2024/merge/oecd_all.xlsx"

writer = pd.ExcelWriter(path, engine = 'xlsxwriter')
out_commodity.to_excel(writer, sheet_name = 'METADATA')
OECD_all.to_excel(writer, sheet_name = 'DATA')
agp_ind_df.to_excel(writer, sheet_name = 'Master_Processed')
writer.save()
writer.close()

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\768851535.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  agp_xe_df = agp_xe_tot.append(agp_xe_nonmps, ignore_index=True)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\768851535.py:25: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  agp_total_df = agp_total.append(agp_total_nonmps, ignore_index=True)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\768851535.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  agpinddfunit['QP_unit'] = 'AG'
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_9612\768851535.py:48: SettingW